# Implementing Language Models are Few-Shot Learners by `Brown et al 2020.`

## Table of Contents

1. [Motivation and Problem Statement](#1-motivation-and-problem-statement)
2. [Core Research Questions](#2-core-research-questions)
3. [Key Concepts and Terminology](#3-key-concepts-and-terminology)
4. [Model Architecture](#4-model-architecture)
5. [Training Data and Preprocessing](#5-training-data-and-preprocessing)
6. [Methodology: In-Context Learning](#6-methodology-in-context-learning)
7. [Mathematical Foundations](#7-mathematical-foundations)
8. [Experimental Setup and Evaluation](#8-experimental-setup-and-evaluation)
9. [Results and Key Findings](#9-results-and-key-findings)
10. [Comparison with Baseline Methods](#10-comparison-with-baseline-methods)
11. [Limitations and Challenges](#11-limitations-and-challenges)

## 1. Problem Statement

### The Problem with Traditional NLP

The standard approach to building NLP systems before GPT-3 followed stages first pre-training then fine tuning and finally building task specific model. In layman terms it will mean suppose you want to build a chatbot for a certain client. Your approach might be as follows:

1. **Pre-train**: Train a large language model on massive amounts of data according to your need (books, websites, articles).
2. **Collect Data**: Gather thousands of labeled examples of customer questions and correct answers.
3. **Fine-tune**: Retrain the model specifically on your customer data.
4. **Deploy**: Use the fine-tuned model.

But when we follow these standard methods there are certain problems such as:
* **Data Hungry**: We need thousands of labeled examples for each new task.
* **Expensive**: Labeling data requires human experts which can be costly and slow.
* **Task-Specific**: Each new task requires a new fine-tuning cycle.
* **Poor Generalization**  Models may not transfer knowledge well to new domains. 
* **Spurious Correlations**  Models might learn dataset specific shortcuts but not have true understanding.

### The Human Connection

When humans learn something they don't need thousands of examples to learn new task. If we are showing a child 3 examples of how a banana and how a mango visually appear they can likely identify one in the the supermarket pretty easily. But fine tuned models need hundreds or thousands of examples.

Therefore the authors answered one of the most important question in this paper i.e. **can we create a model that learns like humans from just a few examples or instructions?**


## 2. Core Research Questions

### Primary Research Question

The major question which this paper answers is that:
> **Can scaling language models to greater sizes unlock the ability to perform new tasks with minimal examples i.e. few shot or no examples at all i.e. zero-shot?**

### Secondary Questions

1. **How does task performance scale with model size?**
   - Do larger models learn more from fewer examples?
   
2. **What is "in-context learning" and how does it work?**
   - Can models learn from examples provided at inference time and without weight updates?
   
3. **What are the limits of few-shot learning?**
   - Which tasks benefit most? Which tasks still require fine-tuning?

## 3. Key Concepts and Terminology

### Understanding the "Shots" in Learning

The paper introduces a  of learning paradigms:

```
┌─────────────────────────────────────────────────────────────────┐
│                     LEARNING PARADIGMS                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌───────────────┐                                              │
│  │  ZERO-SHOT    │  "No examples, just instructions"            │
│  │  Learning     │                                              │
│  │               │  Task: Translate English to French           │
│  │               │  Input: "Hello" → Output: "Bonjour"          │
│  └───────────────┘                                              │
│                                                                 │
│  ┌───────────────┐                                              │
│  │  ONE-SHOT     │  "Single example provided"                   │
│  │  Learning     │                                              │
│  │               │  Example: "Hello" = "Bonjour"                │
│  │               │  Input: "Goodbye" → Output: "Au revoir"      │
│  └───────────────┘                                              │
│                                                                 │
│  ┌───────────────┐                                              │
│  │  FEW-SHOT     │  "Multiple examples (typically 10-100)"      │
│  │  Learning     │                                              │
│  │               │  Examples:                                   │
│  │               │    "Hello" = "Bonjour"                       │
│  │               │    "Thank you" = "Merci"                     │
│  │               │    "Please" = "S'il vous plaît"              │
│  │               │  Input: "How are you?" → Output: ?           │
│  └───────────────┘                                              │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### In-Context Learning

In-context learning is the ability of a language model when we learn from examples provided directly in the input prompt but without any gradient updates to the model's weights.

**Why is this different?**

In traditional machine learning we have training examples with weight updates and then finally model improves but in the **In-Context Learning** we give examples in the prompt itself which then model interprets and then it finally results in generating answers with no weight updates.

**Analogy**: Think of in-context learning like an open-book exam. The book (examples) is given to us during the test and then we use it to answer questions but we don't memorize it for later. We are totally relying on our logical skills to answer our tasks.

#### How In-Context Learning Works

1. **Pattern Matching:**

The model recognizes the pattern `"X" => "Y"` and applies it to every new input.

2. **Internal Fine-tuning:**

Some research suggests the Transformer layers implicitly perform gradient descent on the in-context examples.

3. **Skill Activation**

Pre-training develops latent skills and then the examples activate the relevant skill.


### Meta-learning

The paper says GPT-3 performs a form of **meta-learning** i.e. during pre-training the model develops a broad set of skills and pattern recognition abilities which inference time uses these abilities to rapidly adapt to the task demonstrated in the context.



## 4. Model Architecture

### Building on the Transformer

GPT-3's architecture is based on the **Transformer decoder**, which was introduced in the famous "Attention Is All You Need" paper (2017).

```
┌────────────────────────────────────────────────────────────────┐
│                    GPT-3 ARCHITECTURE                          │
│                                                                │
│                    ┌─────────────────┐                         │
│                    │    Output       │                         │
│                    │  Probabilities  │                         │
│                    └────────┬────────┘                         │
│                             │                                  │
│                    ┌────────▼────────┐                         │
│                    │   Softmax       │                         │
│                    └────────┬────────┘                         │
│                             │                                  │
│                    ┌────────▼────────┐                         │
│                    │  Linear Layer   │                         │
│                    └────────┬────────┘                         │
│                             │                                  │
│              ┌──────────────▼──────────────┐                   │
│              │                             │                   │
│              │   96 Transformer Layers     │ ◄── GPT-3 175B    │
│              │   (Decoder-only blocks)     │                   │
│              │                             │                   │
│              │   Each layer contains:      │                   │
│              │   • Multi-Head Attention    │                   │
│              │   • Feed-Forward Network    │                   │
│              │   • Layer Normalization     │                   │
│              │                             │                   │
│              └──────────────┬──────────────┘                   │
│                             │                                  │
│                    ┌────────▼────────┐                         │
│                    │   Positional    │                         │
│                    │   Encoding      │                         │
│                    └────────┬────────┘                         │
│                             │                                  │
│                    ┌────────▼────────┐                         │
│                    │    Token        │                         │
│                    │   Embeddings    │                         │
│                    └────────┬────────┘                         │
│                             │                                  │
│                    ┌────────▼────────┐                         │
│                    │ Input Tokens    │                         │
│                    └─────────────────┘                         │
│                                                                │
└────────────────────────────────────────────────────────────────┘
```

### Key Difference of GPT-3 Architecture:

In GPT-3 we use sparse attention transformer architecture. The standard transformers have a quadratic bottleneck. Suppose if the sequence length is $n$ then self attention cost is $O(n^2)$ but for GPT-3 we have context length of 2048 and layers and attention heads upto 96. Therefore this becomes extremely expensive in memory and compute.

> A sparse transformer is a transformer where each token attends to only a subset of other tokens but not all of them.

## Complete Implementation from Scratch

### What We'll Implement

1. **Self-Attention:** The core mechanism of GPT.
2. **Transformer Decoder Blocks:** Stacked layers with attention + MLP.
3. **Complete GPT Model:** Complete end-to-end language model.
4. **Training Pipeline:** Next-token prediction.
5. **Few-Shot Learning:** The paper's key innovation.
6. **Visualizations:** Attention patterns and scaling analysis.

In [1]:
import math
import warnings
import numpy as np

In [2]:
import torch
print(torch.__version__)
print(torch.backends.mps.is_available())

/Users/adityamishra/pytorch-test/env/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


2.8.0
True


In [4]:
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [5]:
from dataclasses import dataclass
from typing import Optional, Tuple, List
from torch.utils.data import Dataset, DataLoader

In [6]:
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu
